# 🌍 Geologist Expert Model Training (Stable Version)
This notebook trains Llama-2-7B using standard HuggingFace tools.

**Before running:**
1. Upload `landsat_expert_training_data.jsonl` to the Files panel (left sidebar)
2. Go to Runtime → Change runtime type → Select **T4 GPU**
3. Run all cells!

In [ ]:
# Step 1: Install Dependencies (pinned versions for stability)
!pip install -q transformers==4.36.0 peft==0.7.0 accelerate bitsandbytes datasets trl==0.7.4

In [ ]:
# Step 2: Import Libraries
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Step 3: Configuration
# Using an OPEN model (NO authentication required!)
MODEL_NAME = "NousResearch/Llama-2-7b-hf"
MAX_SEQ_LENGTH = 2048

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
# Step 4: Load Model (4-bit quantized)
print("📦 Loading Llama-2-7B (4-bit)... This takes ~2 minutes")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Model loaded!")

In [ ]:
# Step 5: Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 6: Load Training Data
dataset = load_dataset("json", data_files="landsat_expert_training_data.jsonl", split="train")
print(f"📂 Loaded {len(dataset)} training examples")

# Format prompt
def format_prompt(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"}

dataset = dataset.map(format_prompt)

In [ ]:
# Step 7: Training Arguments
training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_steps=50,
    warmup_steps=5,
    optim="paged_adamw_32bit",
    weight_decay=0.01,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
)

In [ ]:
# Step 8: Create Trainer (using correct API for trl 0.7.x)
tokenizer.model_max_length = MAX_SEQ_LENGTH

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

In [ ]:
# Step 9: TRAIN! 🚀
print("🏋️ Starting training... (10-15 minutes)")
trainer.train()
print("✅ Training complete!")

In [ ]:
# Step 10: Save Model
print("💾 Saving model...")
model.save_pretrained("llama2-geologist-lora")
tokenizer.save_pretrained("llama2-geologist-lora")

# Zip for download
!zip -r llama2-geologist-lora.zip llama2-geologist-lora/
print("\n✅ Done! Download 'llama2-geologist-lora.zip' from the Files panel.")